In [ ]:
# Next-Gen Trading AI (PPO + Transformer) - Google Colab Edition
# ===============================================================
# INSTRUCTIONS FOR COLAB:
# 1. Open Google Colab (https://colab.research.google.com/)
# 2. Go to "Runtime" > "Change runtime type" > Select "T4 GPU"
# 3. Upload 'market_data_1m.csv', 'tradenet_actor.pth' (optional), and 'scaler.pkl' 
#    to the Colab files section.
# 4. Copy-paste this script into a cell block and hit RUN.

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import joblib
import time
import os

# Install external data fetcher if not present in Colab
try:
    import yfinance as yf
except ImportError:
    import subprocess
    print("[INIT] Installing yfinance for automatic global data fetching...")
    subprocess.check_call(["pip", "install", "yfinance"])
    import yfinance as yf

# --- 1. NEXT-GEN SETTINGS & HYPERPARAMETERS ---
SEQ_LENGTH = 60          # Transformer attention window
PROFIT_TARGET = 0.0015   # 0.15% profit target (stricter)
STOP_LOSS = -0.001       # 0.1% stop loss
MAX_HOLD_TIMESTEPS = 20  # Max candles to hold before time decay penalty

# PPO Hyperparameters (The secret sauce of ChatGPT/Gemini)
LR = 0.0003
GAMMA = 0.99             # Discount factor
EPS_CLIP = 0.2           # PPO Clipping parameter (prevents destructive updates)
K_EPOCHS = 4             # How many times to reuse the same batch of experiences
BATCH_SIZE = 512

# Dense Rewards Setup
REWARD_PERFECT = 2.0     # Reached target with zero drawdown
REWARD_OK = 0.5          # Reached target but suffered drawdown
REWARD_TIME_DECAY = -0.5 # Held too long, exited for tiny profit or tiny loss
REWARD_STOP_LOSS = -2.0  # Hit stop loss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INIT] Core System Online. Hardware: {DEVICE}")

# --- 2. AUTOMATIC EXTERNAL DATA SYNC (The "Google" Brain) ---
def fetch_global_context(df_local):
    print("[DATA] Fetching Global Context (India VIX, US Markets) via yfinance...")
    # Get the date range of our localized VPS CSV data
    start_date = df_local.index.min().strftime('%Y-%m-%d')
    # Add one extra day to end_date to reliably grab the final piece of data
    end_date = (df_local.index.max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    
    # We fetch India VIX (fear gauge) and NQ=F (NASDAQ Futures)
    try:
        # Download daily data and forward fill them onto the 1-minute candles
        vix = yf.download("^INDIAVIX", start=start_date, end=end_date, progress=False)['Close']
        nq = yf.download("NQ=F", start=start_date, end=end_date, progress=False)['Close']
        
        # Merge them into the local dataframe based on matching dates
        df_local['date_only'] = df_local.index.date
        
        vix_df = pd.DataFrame(vix).reset_index()
        vix_df.columns = ['date_only', 'india_vix']
        vix_df['date_only'] = vix_df['date_only'].dt.date
        
        nq_df = pd.DataFrame(nq).reset_index()
        nq_df.columns = ['date_only', 'nasdaq_futures']
        nq_df['date_only'] = nq_df['date_only'].dt.date
        
        # Merge
        df_local = df_local.merge(vix_df, on='date_only', how='left')
        df_local = df_local.merge(nq_df, on='date_only', how='left')
        
        # Forward fill weekends/nights, fill remaining with standard values
        df_local['india_vix'] = df_local['india_vix'].ffill().fillna(15.0) 
        df_local['nasdaq_futures'] = df_local['nasdaq_futures'].ffill().fillna(df_local['nasdaq_futures'].mean())
        
        df_local.drop(columns=['date_only'], inplace=True)
        print("[DATA] Global Context Successfully Merged!")
        return df_local
    except Exception as e:
        print(f"[WARN] Could not fetch global data: {e}. Proceeding without it.")
        if 'india_vix' not in df_local: df_local['india_vix'] = 15.0
        if 'nasdaq_futures' not in df_local: df_local['nasdaq_futures'] = 0.0
        return df_local

# --- 3. LOAD LOCAL RESOURCES ---
print("[DATA] Loading local VPS resources...")
# Ensure features match (allow script to auto-fill missing VPS features if any)
import sys

try:
    df = pd.read_csv('market_data_1m.csv')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)
    df.sort_index(inplace=True)
except FileNotFoundError:
    print("❌ market_data_1m.csv not found! Please upload it.")
    sys.exit("Stopping execution: Missing market_data_1m.csv")

# Auto-fetch web data (Only works on Colab or internet-connected machines)
df = fetch_global_context(df)

features = ['close', 'sma_20', 'ema_50', 'rsi_14', 'macd', 'plus_di', 'minus_di', 'adx',
            'kc_upper', 'kc_lower', 'kc_middle', 'volume_shock',
            '15m_ema_50', '15m_sma_20', '15m_rsi_14', '15m_macd', '15m_adx',
            '1h_ema_50', '1h_rsi_14', '1h_adx', 'india_vix', 'nasdaq_futures']

for f in features:
    if f not in df.columns:
        df[f] = 0.0 # Pad zero if missing so architecture remains identical

from sklearn.preprocessing import StandardScaler

# --- 3. SCALER LOGIC (Supports Upgrading) ---
print("[DATA] Configuring Scaler...")
try:
    scaler = joblib.load('scaler.pkl')
    # Check if old scaler matches new feature count
    if hasattr(scaler, 'n_features_in_') and scaler.n_features_in_ != len(features):
        print(f"⚠️ Old scaler has {scaler.n_features_in_} features. New brain needs {len(features)}.")
        print("🔄 Fitting a fresh Scaler for the Next-Gen Brain...")
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(df[features].values)
    else:
        print("✅ Matching scaler found. Transforming data...")
        data_scaled = scaler.transform(df[features].values)
except Exception as e:
    print(f"⚠️ Scaler not found or invalid ({e}). Fitting a fresh one...")
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(df[features].values)

# Save the new scaler so it can be downloaded back to VPS
joblib.dump(scaler, 'scaler_ppo.pkl')
print("💾 Saved new scaler -> scaler_ppo.pkl")

INPUT_DIM = len(features)

# --- 4. ADVANCED ARCHITECTURE: TRANSFORMER ACTOR-CRITIC ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)  # [1, max_len, d_model]

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        device = x.device
        x = x + self.pe[:, :x.size(1), :].to(device)
        return x

class PPO_Transformer(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=2):
        super(PPO_Transformer, self).__init__()
        self.feature_extractor = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layers = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        # PPO requires two heads: Action (Actor) and Value (Critic)
        # Action space: 0 (Hold), 1 (Buy) -> We use softmax for probabilities
        self.actor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=-1)
        )
        
        # Critic estimates "how good is this state?" for baseline calculations
        self.critic = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        # x: [batch, seq_len, features]
        x = self.feature_extractor(x)
        x = self.pos_encoder(x)
        
        encoded = self.transformer(x)
        # Take the output of the last sequence step
        final_state = encoded[:, -1, :] 
        
        action_probs = self.actor(final_state)
        state_value = self.critic(final_state)
        return action_probs, state_value

print("[MODEL] Booting Next-Gen Transformer PPO...")
model = PPO_Transformer(input_dim=INPUT_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Optional: Load old weights if upgrading (Requires surgery, but possible)
# For this script we assume training fresh or loading an already saved Transformer PPO

# --- 5. PPO MEMORY BUFFER ---
class RolloutBuffer:
    def __init__(self):
        self.actions = []
        self.states = []
        self.logprobs = []
        self.rewards = []
        self.values = []
        self.is_terminals = []
    
    def clear(self):
        del self.actions[:]
        del self.states[:]
        del self.logprobs[:]
        del self.rewards[:]
        del self.values[:]
        del self.is_terminals[:]

# --- 6. PROXIMAL POLICY OPTIMIZATION (PPO) CORE ---
def update_ppo(buffer):
    # Convert lists to tensors
    old_states = torch.stack(buffer.states).squeeze(1).detach().to(DEVICE)
    old_actions = torch.stack(buffer.actions).detach().to(DEVICE)
    old_logprobs = torch.stack(buffer.logprobs).detach().to(DEVICE)
    old_values = torch.stack(buffer.values).squeeze().detach().to(DEVICE)
    
    rewards = buffer.rewards
    is_terminals = buffer.is_terminals
    
    # Calculate GAE (Generalized Advantage Estimation) or standard Returns
    returns = []
    discounted_reward = 0
    for reward, is_terminal in zip(reversed(rewards), reversed(is_terminals)):
        if is_terminal:
            discounted_reward = 0
        discounted_reward = reward + (GAMMA * discounted_reward)
        returns.insert(0, discounted_reward)
        
    returns = torch.tensor(returns, dtype=torch.float32).to(DEVICE)
    returns = (returns - returns.mean()) / (returns.std() + 1e-7)
    
    advantages = returns - old_values
    
    # Optimize policy for K epochs (PPO reuse mechanism)
    for _ in range(K_EPOCHS):
        # Forward pass all saved states
        action_probs, state_values = model(old_states)
        dist = Categorical(action_probs)
        logprobs = dist.log_prob(old_actions)
        dist_entropy = dist.entropy()
        
        # Calculate ratio: pi_theta / pi_theta__old
        ratios = torch.exp(logprobs - old_logprobs)
        
        # Surrogate Loss (Clipping)
        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1-EPS_CLIP, 1+EPS_CLIP) * advantages
        
        # Final PPO Loss
        loss = -torch.min(surr1, surr2) + 0.5*nn.MSELoss()(state_values.squeeze(), returns) - 0.01*dist_entropy
        
        # Gradient Update
        optimizer.zero_grad()
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        

# --- 7. THE LEARNING LOOP ---
print("[TRAIN] Initiating PPO Dense Reward Simulation...")
buffer = RolloutBuffer()
epochs = 100 # ADVANCED: 100 Epochs for 1-2 day fully automated training on Colab
update_timestep = BATCH_SIZE

for epoch in range(1, epochs + 1):
    total_reward = 0
    trade_count = 0
    wins = 0
    
    i = SEQ_LENGTH
    
    while i < len(data_scaled) - MAX_HOLD_TIMESTEPS - 1:
        # 1. Observe State
        state = data_scaled[i-SEQ_LENGTH : i]
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        
        # 2. Select Action using PPO Policy
        model.eval()
        with torch.no_grad():
            action_probs, state_value = model(state_tensor)
            dist = Categorical(action_probs)
            action = dist.sample()
            
        buffer.states.append(state_tensor)
        buffer.actions.append(action)
        buffer.logprobs.append(dist.log_prob(action))
        buffer.values.append(state_value)
        
        # 3. Simulate Environment (Take Action)
        if action.item() == 1: # AI chose to BUY
            trade_count += 1
            entry = df['close'].iloc[i]
            
            reward = 0
            timesteps_held = 0
            drawdown_occurred = False
            
            for f in range(1, MAX_HOLD_TIMESTEPS + 1):
                future_idx = i + f
                future_price = df['close'].iloc[future_idx]
                pct = (future_price - entry) / entry
                
                if pct < 0:
                    drawdown_occurred = True
                    
                if pct >= PROFIT_TARGET:
                    if not drawdown_occurred:
                        reward = REWARD_PERFECT
                    else:
                        reward = REWARD_OK
                    wins += 1
                    timesteps_held = f
                    break
                elif pct <= STOP_LOSS:
                    reward = REWARD_STOP_LOSS
                    timesteps_held = f
                    break
                    
            if reward == 0:
                reward = REWARD_TIME_DECAY # Too slow
                timesteps_held = MAX_HOLD_TIMESTEPS
                
            buffer.rewards.append(reward)
            buffer.is_terminals.append(True) # Trade over, episode ends
            
            total_reward += reward
            i += timesteps_held # Jump forward in time
            
        else:
            # AI chose to IDLE
            # No penalty for idle, keeps it safe.
            buffer.rewards.append(0)
            buffer.is_terminals.append(False)
            i += 1
            
        # 4. Update PPO Network
        if len(buffer.states) >= update_timestep:
            model.train()
            update_ppo(buffer)
            buffer.clear()

    # End of epoch summary
    win_rate = (wins / trade_count * 100) if trade_count > 0 else 0
    print(f"Epoch {epoch} | Trades: {trade_count} | Wins: {wins} | Win Rate: {win_rate:.1f}% | Total PPO Reward: {total_reward:.1f}")

# Save the super brain
checkpoint = {
    'input_size': INPUT_DIM,
    'model_state': model.state_dict(),
    'features': features
}
torch.save(checkpoint, 'tradenet_ppo_transformer.pth')
print("✅ Next-Gen PPO Transformer Saved -> tradenet_ppo_transformer.pth")
